# 🏢 ML Challenge 2026: Business Entity Resolution

## Complete Pipeline — Blocking → Feature Engineering → XGBoost Matching

This notebook implements an end-to-end entity resolution pipeline that:
1. **Preprocesses** names & addresses (transliteration, normalization, abbreviation expansion)
2. **Blocks** candidates using multi-strategy blocking (TF-IDF, phonetic, country-partitioned)
3. **Engineers features** (Jaccard, Levenshtein, Jaro-Winkler, TF-IDF cosine, token overlap, etc.)
4. **Trains an XGBoost classifier** optimised for F₀.₅
5. **Generates** `matching_results.tsv` and `candidate_pairs.tsv`

### Edge Cases Handled
- **Unseen country (France)** in test — no hard-coded country lists
- **Transliterated names** (Hindi, Tamil, Kannada → Latin)
- **Singletons** — correctly predicting no-match scores full 1.0
- **Missing/empty addresses**
- **DBA / trade names** completely different from legal name
- **Address reordering** (City, State, Street vs Street, City, State)
- **Abbreviation variations** (Corp/Corporation, Pvt/Private, Ltd/Limited, Rd/Road, St/Street)

---

## 0. Setup & Install Dependencies

In [ ]:
# Packages are installed in your local virtual environment (see setup steps),
# not inside the notebook. If you skipped that step, uncomment the line below:
# !pip install -q xgboost scikit-learn pandas numpy tqdm python-Levenshtein rapidfuzz unidecode jellyfish

print('Using packages from the active kernel/venv.')


In [ ]:
import os
import re
import gc
import subprocess
import warnings
from collections import defaultdict
from typing import Dict, List, Set, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

import xgboost as xgb

import Levenshtein
from rapidfuzz import fuzz
import jellyfish
from unidecode import unidecode

warnings.filterwarnings('ignore')
tqdm.pandas()

print('All imports successful!')

## 1. Upload & Load Data

Upload the `student_resource/dataset/` folder to your Google Drive.  
Adjust `DATA_DIR` below to match where you placed the files.

In [ ]:
# ============================================================
# LOCAL: dataset/ folder sits alongside this notebook, with train/ and test/ subfolders
# ============================================================
DATA_DIR = 'dataset'

# Sanity check
import os as _os
print(_os.listdir(DATA_DIR))


In [ ]:
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'test')

# Output directory (relative to notebook location)
OUTPUT_DIR = './output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Train dir: {TRAIN_DIR}')
print(f'Test dir:  {TEST_DIR}')
print(f'Output dir: {OUTPUT_DIR}')


In [ ]:
%%time
# Load training data
print('Loading training data...')
train_s1 = pd.read_csv(os.path.join(TRAIN_DIR, 'train_source1.tsv'), sep='\t', dtype=str)
train_s2 = pd.read_csv(os.path.join(TRAIN_DIR, 'train_source2.tsv'), sep='\t', dtype=str)
train_s3 = pd.read_csv(os.path.join(TRAIN_DIR, 'train_source3.tsv'), sep='\t', dtype=str)
train_gt = pd.read_csv(os.path.join(TRAIN_DIR, 'train_ground_truth.tsv'), sep='\t', dtype=str)

print(f'Train S1: {len(train_s1):,} records')
print(f'Train S2: {len(train_s2):,} records')
print(f'Train S3: {len(train_s3):,} records')
print(f'Ground truth: {len(train_gt):,} entries')

# Load test data
print('\nLoading test data...')
test_s1 = pd.read_csv(os.path.join(TEST_DIR, 'test_source1.tsv'), sep='\t', dtype=str)
test_s2 = pd.read_csv(os.path.join(TEST_DIR, 'test_source2.tsv'), sep='\t', dtype=str)
test_s3 = pd.read_csv(os.path.join(TEST_DIR, 'test_source3.tsv'), sep='\t', dtype=str)

print(f'Test S1:  {len(test_s1):,} records')
print(f'Test S2:  {len(test_s2):,} records')
print(f'Test S3:  {len(test_s3):,} records')

In [ ]:
# Quick data preview
print('=== Source 1 (Reference) ===')
display(train_s1.head(3))
print('\n=== Source 2 ===')
display(train_s2.head(3))
print('\n=== Source 3 ===')
display(train_s3.head(3))
print('\n=== Ground Truth ===')
display(train_gt.head(5))

# Check countries in test
print('\nCountries in train:', train_s1['country'].unique())
print('Countries in test: ', test_s1['country'].unique())

## 2. Text Preprocessing & Normalization

This is critical for entity resolution — we need to handle:
- Unicode transliteration (Hindi/Tamil/Kannada -> Latin)
- Legal suffix normalization (Corp -> Corporation, Pvt -> Private, etc.)
- Address abbreviation expansion (Rd -> Road, St -> Street, etc.)
- Punctuation & whitespace normalization

In [ ]:
# ============================================================
# Business name normalization
# ============================================================

# Legal suffix mappings - comprehensive set
LEGAL_SUFFIX_MAP = {
    # English
    'corp': 'corporation', 'inc': 'incorporated', 'ltd': 'limited',
    'llc': 'limited liability company', 'llp': 'limited liability partnership',
    'co': 'company', 'assoc': 'associates', 'intl': 'international',
    'natl': 'national', 'svcs': 'services', 'svc': 'service',
    'mfg': 'manufacturing', 'mgmt': 'management', 'grp': 'group',
    'hldg': 'holding', 'hldgs': 'holdings', 'ent': 'enterprise',
    'tech': 'technology', 'techs': 'technologies',
    'sys': 'systems', 'sol': 'solutions', 'soln': 'solutions',
    'solns': 'solutions', 'eng': 'engineering', 'engr': 'engineering',
    'acct': 'accounting', 'assn': 'association',
    'dept': 'department', 'dist': 'distribution',
    'fdn': 'foundation', 'fndn': 'foundation',
    # India-specific
    'pvt': 'private', 'pvtltd': 'private limited',
    # French
    'sarl': 'sarl', 'sas': 'sas', 'sa': 'sa', 'eurl': 'eurl',
    'ste': 'societe', 'cie': 'compagnie',
}

# Address abbreviation mappings
ADDRESS_ABBREV_MAP = {
    'st': 'street', 'rd': 'road', 'dr': 'drive', 'ave': 'avenue',
    'blvd': 'boulevard', 'ln': 'lane', 'ct': 'court', 'pl': 'place',
    'pkwy': 'parkway', 'hwy': 'highway', 'cir': 'circle',
    'trl': 'trail', 'ter': 'terrace', 'sq': 'square',
    'apt': 'apartment', 'ste': 'suite', 'bldg': 'building',
    'fl': 'floor', 'flr': 'floor', 'rm': 'room',
    'n': 'north', 's': 'south', 'e': 'east', 'w': 'west',
    'ne': 'northeast', 'nw': 'northwest', 'se': 'southeast', 'sw': 'southwest',
    'opp': 'opposite', 'nr': 'near', 'adj': 'adjacent',
    'po': 'post office',
    # India-specific
    'dist': 'district', 'vill': 'village', 'tal': 'taluka',
    'teh': 'tehsil',
    # French
    'av': 'avenue', 'bd': 'boulevard',
    'rte': 'route', 'chem': 'chemin', 'imp': 'impasse',
}

print(f'Legal suffixes: {len(LEGAL_SUFFIX_MAP)}')
print(f'Address abbreviations: {len(ADDRESS_ABBREV_MAP)}')

In [ ]:
def transliterate_to_latin(text):
    """Convert non-Latin scripts (Hindi, Tamil, Kannada, etc.) to Latin.
    Uses unidecode for broad Unicode -> ASCII transliteration.
    """
    if not text or pd.isna(text):
        return ''
    try:
        text.encode('ascii')
        return text  # Already ASCII
    except UnicodeEncodeError:
        return unidecode(text)


def normalize_business_name(name):
    """Normalize a business name for matching."""
    if not name or pd.isna(name):
        return ''

    # Transliterate non-Latin scripts
    name = transliterate_to_latin(name)

    # Lowercase
    name = name.lower().strip()

    # Remove URLs (e.g., www.something.com, website.com)
    name = re.sub(r'\|?\s*(?:https?://)?(?:www\.)?[a-z0-9.-]+\.[a-z]{2,}', '', name)

    # Remove common prefixes/noise
    name = re.sub(r'^(the|a|an)\s+', '', name)
    name = re.sub(r'^[<#]+\s*', '', name)  # Remove leading << or # symbols
    name = re.sub(r'^--\s*', '', name)      # Remove leading --

    # Normalize & to 'and'
    name = name.replace('&', ' and ')

    # Remove punctuation except hyphens
    name = re.sub(r"[.'\",;:!?()\[\]{}@#$%^*+=~`/\\|]", ' ', name)

    # Expand legal suffixes
    tokens = name.split()
    expanded = []
    for token in tokens:
        cleaned = token.strip('.')
        if cleaned in LEGAL_SUFFIX_MAP:
            expanded.append(LEGAL_SUFFIX_MAP[cleaned])
        else:
            expanded.append(token)
    name = ' '.join(expanded)

    # Remove 'dba' (doing business as) and split
    name = re.sub(r'\bdba\b', ' ', name)

    # Collapse whitespace
    name = re.sub(r'\s+', ' ', name).strip()

    return name


def normalize_address(address):
    """Normalize a business address for matching."""
    if not address or pd.isna(address):
        return ''

    # Transliterate non-Latin scripts
    address = transliterate_to_latin(address)

    # Lowercase
    address = address.lower().strip()

    # Remove unit/apartment/suite prefixes and their identifiers
    address = re.sub(
        r'\b(?:unit|apt|ste|suite|apartment|room|rm|bldg|building|floor|fl|flr)'
        r'\s*[#.]?\s*\w*', ' ', address)

    # Remove PO Box references
    address = re.sub(r'\bp\.?o\.?\s*box\s*\d*', ' ', address)

    # Remove 'null' strings
    address = re.sub(r'\bnull\b', ' ', address)
    address = re.sub(r'\bnan\b', ' ', address)

    # Normalize punctuation
    address = address.replace('&', ' and ')
    address = re.sub(r"[.'\",;:!?()\[\]{}@#$%^*+=~`/\\|]", ' ', address)

    # Expand address abbreviations
    tokens = address.split()
    expanded = []
    for token in tokens:
        cleaned = token.strip('.')
        if cleaned in ADDRESS_ABBREV_MAP:
            expanded.append(ADDRESS_ABBREV_MAP[cleaned])
        else:
            expanded.append(token)
    address = ' '.join(expanded)

    # Remove ordinal suffixes from numbers (1st, 2nd, 3rd, etc.)
    address = re.sub(r'(\d+)(?:st|nd|rd|th)\b', r'\1', address)

    # Collapse whitespace
    address = re.sub(r'\s+', ' ', address).strip()

    return address


def extract_tokens(text):
    """Extract a set of meaningful tokens from normalized text."""
    if not text:
        return set()
    return {t for t in text.split() if len(t) > 1 or t.isdigit()}


def extract_numbers(text):
    """Extract all numeric sequences from text (useful for address matching)."""
    if not text:
        return set()
    return set(re.findall(r'\d+', text))


# Test normalization
test_cases = [
    ('Raj Investments LLP', '6(29), C.I.T. Colony, 2Nd Main Road Mylapore, Chennai, Tamil Nadu'),
    ('Dahlia Power Reliable Scientific LLC', '630 45th Terrace, Kansas City, MO'),
    ('Custom Wealth Services LLC', 'OH, Columbus, 5559 Orville Avenue'),
]

for name, addr in test_cases:
    print(f'Name: {name!r} -> {normalize_business_name(name)!r}')
    print(f'Addr: {addr!r} -> {normalize_address(addr)!r}')
    print()

## 3. Preprocess All Sources

In [ ]:
# ============================================================
# Precompiled regex patterns (compiled once, reused for every row/column
# instead of being re-parsed inside a Python function called per row)
# ============================================================
_URL_RE = re.compile(r'\|?\s*(?:https?://)?(?:www\.)?[a-z0-9.-]+\.[a-z]{2,}')
_LEAD_ARTICLE_RE = re.compile(r'^(the|a|an)\s+')
_LEAD_SYMBOL_RE = re.compile(r'^[<#]+\s*')
_LEAD_DASH_RE = re.compile(r'^--\s*')
_PUNCT_RE = re.compile(r"[.\'\",;:!?()\[\]{}@#$%^*+=~`/\\|]")
_DBA_RE = re.compile(r'\bdba\b')
_WS_RE = re.compile(r'\s+')
_NAME_SUFFIX_RE = re.compile(r'\b(' + '|'.join(re.escape(k) for k in LEGAL_SUFFIX_MAP) + r')\b')

_UNIT_RE = re.compile(
    r'\b(?:unit|apt|ste|suite|apartment|room|rm|bldg|building|floor|fl|flr)\s*[#.]?\s*\w*')
_POBOX_RE = re.compile(r'\bp\.?o\.?\s*box\s*\d*')
_NULLNAN_RE = re.compile(r'\bnull\b|\bnan\b')
_ORDINAL_RE = re.compile(r'(\d+)(?:st|nd|rd|th)\b')
_ADDR_ABBREV_RE = re.compile(r'\b(' + '|'.join(re.escape(k) for k in ADDRESS_ABBREV_MAP) + r')\b')


def _expand_name_suffix(m):
    return LEGAL_SUFFIX_MAP[m.group(0)]

def _expand_addr_abbrev(m):
    return ADDRESS_ABBREV_MAP[m.group(0)]


def normalize_business_name_vec(s):
    """Vectorized name normalization - runs each regex pass once over the
    WHOLE column instead of calling a Python function per row."""
    s = s.fillna('').apply(transliterate_to_latin)  # unavoidable row-wise (unidecode)
    s = s.str.lower().str.strip()
    s = s.str.replace(_URL_RE, '', regex=True)
    s = s.str.replace(_LEAD_ARTICLE_RE, '', regex=True)
    s = s.str.replace(_LEAD_SYMBOL_RE, '', regex=True)
    s = s.str.replace(_LEAD_DASH_RE, '', regex=True)
    s = s.str.replace('&', ' and ', regex=False)
    s = s.str.replace(_PUNCT_RE, ' ', regex=True)
    s = s.str.replace(_NAME_SUFFIX_RE, _expand_name_suffix, regex=True)
    s = s.str.replace(_DBA_RE, ' ', regex=True)
    s = s.str.replace(_WS_RE, ' ', regex=True).str.strip()
    return s


def normalize_address_vec(s):
    """Vectorized address normalization."""
    s = s.fillna('').apply(transliterate_to_latin)
    s = s.str.lower().str.strip()
    s = s.str.replace(_UNIT_RE, ' ', regex=True)
    s = s.str.replace(_POBOX_RE, ' ', regex=True)
    s = s.str.replace(_NULLNAN_RE, ' ', regex=True)
    s = s.str.replace('&', ' and ', regex=False)
    s = s.str.replace(_PUNCT_RE, ' ', regex=True)
    s = s.str.replace(_ADDR_ABBREV_RE, _expand_addr_abbrev, regex=True)
    s = s.str.replace(_ORDINAL_RE, r'\1', regex=True)
    s = s.str.replace(_WS_RE, ' ', regex=True).str.strip()
    return s


def preprocess_source(df, desc=''):
    """Add normalized columns to a source DataFrame - vectorized, memory-lean."""
    # No df.copy() here - mutate in place; the caller reassigns the variable
    # (e.g. train_s1 = preprocess_source(train_s1, ...)) so the old object
    # still gets garbage collected, but we don't pay for two full copies at once.
    df['business_name'] = df['business_name'].fillna('')
    df['business_address'] = df['business_address'].fillna('')
    df['country'] = df['country'].fillna('Unknown')

    print(f'  Normalizing business names ({desc})...')
    df['name_norm'] = normalize_business_name_vec(df['business_name'])

    print(f'  Normalizing addresses ({desc})...')
    df['addr_norm'] = normalize_address_vec(df['business_address'])

    df['country_lower'] = df['country'].str.lower().str.strip()

    print(f'  Extracting tokens ({desc})...')
    df['name_tokens'] = df['name_norm'].apply(extract_tokens)
    df['addr_tokens'] = df['addr_norm'].apply(extract_tokens)
    df['addr_numbers'] = df['addr_norm'].apply(extract_numbers)

    df['combined'] = df['name_norm'] + ' ' + df['addr_norm']

    df['name_metaphone'] = df['name_norm'].apply(
        lambda x: jellyfish.metaphone(x[:50]) if x else ''
    )
    df['name_prefix'] = df['name_norm'].str[:5].fillna('')

    # Nothing downstream uses the raw text columns once name_norm/addr_norm
    # exist (final output only needs entity_id) - drop them to free memory.
    df.drop(columns=['business_name', 'business_address'], inplace=True)

    return df

print('Preprocessing will take a few minutes per source due to dataset size...')


In [ ]:
%%time
# Preprocess training sources
print('=== Preprocessing Training Sources ===')
train_s1 = preprocess_source(train_s1, 'train_s1'); gc.collect()
train_s2 = preprocess_source(train_s2, 'train_s2'); gc.collect()
train_s3 = preprocess_source(train_s3, 'train_s3'); gc.collect()
print('Training preprocessing complete!')


In [ ]:
%%time
# Preprocess test sources
print('=== Preprocessing Test Sources ===')
test_s1 = preprocess_source(test_s1, 'test_s1'); gc.collect()
test_s2 = preprocess_source(test_s2, 'test_s2'); gc.collect()
test_s3 = preprocess_source(test_s3, 'test_s3'); gc.collect()
print('Test preprocessing complete!')


## 4. Parse Ground Truth

In [ ]:
def parse_ground_truth(gt_df):
    """Parse ground truth into {s1_id: set(matched_ids)}."""
    gt_map = {}
    for _, row in tqdm(gt_df.iterrows(), total=len(gt_df), desc='Parsing GT'):
        s1_id = row['source1_entity_id']
        matched = row.get('matched_entity_ids', '')
        if pd.isna(matched) or str(matched).strip() == '':
            gt_map[s1_id] = set()
        else:
            gt_map[s1_id] = set(str(matched).strip().split(','))
    return gt_map

gt_map = parse_ground_truth(train_gt)

n_singletons = sum(1 for v in gt_map.values() if len(v) == 0)
n_with_matches = sum(1 for v in gt_map.values() if len(v) > 0)
print(f'\nSingletons (no match): {n_singletons:,}')
print(f'With matches: {n_with_matches:,}')
print(f'Avg matches per entity: {np.mean([len(v) for v in gt_map.values() if len(v) > 0]):.2f}')

## 5. Blocking / Candidate Generation

We cannot compare every S1 entity against every S2/S3 entity (trillions of pairs).  
**Multi-strategy blocking** reduces the candidate set while maintaining high recall:

1. **Country filter** — only compare entities from the same country
2. **TF-IDF cosine similarity** on combined (name + address) text
3. **Phonetic blocking** — Metaphone key on business name
4. **Name prefix blocking** — first 5 chars of normalized name

In [ ]:
# ============================================================
# CONFIGURATION - tune these for recall vs. speed tradeoff
# ============================================================
TFIDF_TOP_K = 20           # Top-K TF-IDF candidates per S1 entity
TFIDF_MIN_SCORE = 0.15     # Minimum cosine similarity threshold
BLOCKING_BATCH_SIZE = 5000 # Process S1 entities in batches for memory
MAX_CANDIDATES_PER_S1 = 50 # Cap candidates to avoid memory blowup

# For training, we use a subset to build the model efficiently
# Set to None to use all data (will be very slow)
TRAIN_SAMPLE_SIZE = 50000  # Number of S1 entities to sample for training

print(f'TF-IDF top-K: {TFIDF_TOP_K}')
print(f'TF-IDF min score: {TFIDF_MIN_SCORE}')
print(f'Max candidates per S1: {MAX_CANDIDATES_PER_S1}')
print(f'Train sample size: {TRAIN_SAMPLE_SIZE}')

In [ ]:
def build_blocking_index(s2_df, s3_df):
    """Build inverted indices for blocking on S2+S3 combined."""

    # Combine S2 and S3
    candidates = pd.concat([s2_df, s3_df], ignore_index=True)

    index = {
        'df': candidates,
        'id_to_idx': {eid: i for i, eid in enumerate(candidates['entity_id'])},
        'metaphone': defaultdict(list),
        'prefix': defaultdict(list),
        'country': defaultdict(list),
    }

    for i, row in tqdm(candidates.iterrows(), total=len(candidates), desc='Building index'):
        eid = row['entity_id']
        country = row['country_lower']

        # Country index
        index['country'][country].append(i)

        # Metaphone index
        mph = row['name_metaphone']
        if mph:
            index['metaphone'][(country, mph)].append(i)

        # Name prefix index
        prefix = row['name_prefix']
        if prefix:
            index['prefix'][(country, prefix)].append(i)

    return index

print('Building blocking index for training data...')

In [ ]:
%%time
train_block_index = build_blocking_index(train_s2, train_s3)

In [ ]:
def tfidf_blocking(s1_df, candidates_df, country, top_k=TFIDF_TOP_K,
                   min_score=TFIDF_MIN_SCORE):
    """TF-IDF cosine-similarity blocking within a country partition.

    Returns {s1_id: [(candidate_id, score), ...]}.
    """
    s1_country = s1_df[s1_df['country_lower'] == country]
    cand_country = candidates_df[candidates_df['country_lower'] == country]

    if len(s1_country) == 0 or len(cand_country) == 0:
        return {}

    print(f'  TF-IDF blocking for country={country!r}: '
          f'{len(s1_country):,} S1 x {len(cand_country):,} candidates')

    # Fit TF-IDF on combined texts
    vectorizer = TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(2, 4),
        max_features=100000,
        sublinear_tf=True,
        dtype=np.float32,
    )

    all_texts = pd.concat([
        s1_country['combined'],
        cand_country['combined']
    ])
    vectorizer.fit(all_texts)

    cand_vectors = vectorizer.transform(cand_country['combined'])
    cand_ids = cand_country['entity_id'].values

    results = {}
    s1_ids = s1_country['entity_id'].values
    s1_texts = s1_country['combined'].values

    # Process in batches to manage memory
    batch_size = BLOCKING_BATCH_SIZE
    for batch_start in tqdm(range(0, len(s1_ids), batch_size),
                            desc=f'TF-IDF {country}'):
        batch_end = min(batch_start + batch_size, len(s1_ids))
        batch_texts = s1_texts[batch_start:batch_end]
        batch_s1_ids = s1_ids[batch_start:batch_end]

        s1_vectors = vectorizer.transform(batch_texts)
        sim_matrix = cosine_similarity(s1_vectors, cand_vectors)

        for i, s1_id in enumerate(batch_s1_ids):
            scores = sim_matrix[i]
            # Get top-K above threshold
            top_indices = np.argsort(scores)[::-1][:top_k]
            candidates = []
            for idx in top_indices:
                if scores[idx] >= min_score:
                    candidates.append((cand_ids[idx], float(scores[idx])))
            if candidates:
                results[s1_id] = candidates

    del cand_vectors
    gc.collect()

    return results

In [ ]:
def multi_strategy_blocking(s1_df, s2_df, s3_df, block_index):
    """Apply multi-strategy blocking to generate candidate pairs.

    Combines:
      1. TF-IDF cosine similarity (within country partition)
      2. Phonetic (Metaphone) key matching
      3. Name prefix matching

    Returns {s1_id: set(candidate_ids)}.
    """
    candidates_df = block_index['df']
    candidate_map = defaultdict(set)

    # Strategy 1: TF-IDF blocking by country
    countries = s1_df['country_lower'].unique()
    print(f'Countries to process: {countries}')

    for country in countries:
        tfidf_results = tfidf_blocking(s1_df, candidates_df, country)
        for s1_id, cands in tfidf_results.items():
            for cand_id, score in cands:
                candidate_map[s1_id].add(cand_id)

    print(f'After TF-IDF: {sum(len(v) for v in candidate_map.values()):,} pairs')

    # Strategy 2: Phonetic blocking
    for _, row in tqdm(s1_df.iterrows(), total=len(s1_df), desc='Phonetic blocking'):
        s1_id = row['entity_id']
        country = row['country_lower']
        mph = row['name_metaphone']

        if mph:
            key = (country, mph)
            if key in block_index['metaphone']:
                for idx in block_index['metaphone'][key]:
                    cand_id = candidates_df.iloc[idx]['entity_id']
                    candidate_map[s1_id].add(cand_id)

    print(f'After Phonetic: {sum(len(v) for v in candidate_map.values()):,} pairs')

    # Strategy 3: Name prefix blocking
    for _, row in tqdm(s1_df.iterrows(), total=len(s1_df), desc='Prefix blocking'):
        s1_id = row['entity_id']
        country = row['country_lower']
        prefix = row['name_prefix']

        if prefix:
            key = (country, prefix)
            if key in block_index['prefix']:
                for idx in block_index['prefix'][key][:100]:  # cap per key
                    cand_id = candidates_df.iloc[idx]['entity_id']
                    candidate_map[s1_id].add(cand_id)

    print(f'After Prefix: {sum(len(v) for v in candidate_map.values()):,} pairs')

    # Cap candidates per S1
    for s1_id in candidate_map:
        if len(candidate_map[s1_id]) > MAX_CANDIDATES_PER_S1:
            candidate_map[s1_id] = set(list(candidate_map[s1_id])[:MAX_CANDIDATES_PER_S1])

    total_pairs = sum(len(v) for v in candidate_map.values())
    n_with_cands = sum(1 for v in candidate_map.values() if len(v) > 0)
    print(f'\nFinal: {total_pairs:,} candidate pairs for {n_with_cands:,} S1 entities')
    print(f'S1 entities with no candidates: {len(s1_df) - n_with_cands:,}')

    return dict(candidate_map)

In [ ]:
# Sample S1 entities for training (full dataset is too large for Colab)
if TRAIN_SAMPLE_SIZE and TRAIN_SAMPLE_SIZE < len(train_s1):
    # Stratified sample: ensure we get both singletons and matched entities
    s1_ids_with_match = [sid for sid, v in gt_map.items() if len(v) > 0]
    s1_ids_singleton = [sid for sid, v in gt_map.items() if len(v) == 0]

    # Keep 10% singletons in sample (mirrors ~5.6% in real data)
    n_singletons_sample = int(TRAIN_SAMPLE_SIZE * 0.10)
    n_matched_sample = TRAIN_SAMPLE_SIZE - n_singletons_sample

    np.random.seed(42)
    sampled_matched = np.random.choice(
        s1_ids_with_match,
        size=min(n_matched_sample, len(s1_ids_with_match)),
        replace=False)
    sampled_singletons = np.random.choice(
        s1_ids_singleton,
        size=min(n_singletons_sample, len(s1_ids_singleton)),
        replace=False)

    sampled_ids = set(sampled_matched) | set(sampled_singletons)
    train_s1_sample = train_s1[train_s1['entity_id'].isin(sampled_ids)].copy()
    print(f'Sampled {len(train_s1_sample):,} S1 entities for training')
else:
    train_s1_sample = train_s1.copy()
    print(f'Using all {len(train_s1_sample):,} S1 entities')

In [ ]:
%%time
print('=== Running Multi-Strategy Blocking on Training Sample ===')
train_candidates = multi_strategy_blocking(
    train_s1_sample, train_s2, train_s3, train_block_index
)

In [ ]:
# Evaluate blocking recall on training sample
def evaluate_blocking_recall(candidates, gt):
    """Check how many true matches are captured by blocking."""
    total_true = 0
    total_captured = 0
    missed_examples = []

    for s1_id in candidates:
        if s1_id not in gt:
            continue
        true_matches = gt[s1_id]
        predicted_cands = candidates.get(s1_id, set())

        total_true += len(true_matches)
        captured = true_matches & predicted_cands
        total_captured += len(captured)

        missed = true_matches - predicted_cands
        if missed and len(missed_examples) < 5:
            missed_examples.append((s1_id, missed))

    recall = total_captured / total_true if total_true > 0 else 0
    print(f'Blocking Recall: {recall:.4f} ({total_captured:,}/{total_true:,})')
    print(f'Avg candidates per S1: {np.mean([len(v) for v in candidates.values()]):.1f}')

    if missed_examples:
        print(f'\nExamples of missed true matches:')
        for s1_id, missed in missed_examples[:3]:
            print(f'  {s1_id}: missed {missed}')

    return recall

blocking_recall = evaluate_blocking_recall(train_candidates, gt_map)

## 6. Feature Engineering

For each (S1, candidate) pair, compute a rich set of similarity features.

In [ ]:
def jaccard_similarity(set_a, set_b):
    """Jaccard similarity between two sets."""
    if not set_a and not set_b:
        return 1.0  # Both empty -> 'same' (edge case: both missing)
    if not set_a or not set_b:
        return 0.0
    intersection = set_a & set_b
    union = set_a | set_b
    return len(intersection) / len(union)


def overlap_coefficient(set_a, set_b):
    """Overlap coefficient - good for subset matching."""
    if not set_a or not set_b:
        return 0.0
    intersection = set_a & set_b
    return len(intersection) / min(len(set_a), len(set_b))


def containment_score(set_a, set_b):
    """How much of set_a is contained in set_b."""
    if not set_a:
        return 0.0
    return len(set_a & set_b) / len(set_a)


def compute_pair_features(s1_row, cand_row):
    """Compute similarity features for a (S1, candidate) pair."""
    features = {}

    name1 = s1_row['name_norm']
    name2 = cand_row['name_norm']
    addr1 = s1_row['addr_norm']
    addr2 = cand_row['addr_norm']

    name_tokens1 = s1_row['name_tokens']
    name_tokens2 = cand_row['name_tokens']
    addr_tokens1 = s1_row['addr_tokens']
    addr_tokens2 = cand_row['addr_tokens']
    addr_nums1 = s1_row['addr_numbers']
    addr_nums2 = cand_row['addr_numbers']

    # ==========================================
    # NAME FEATURES
    # ==========================================

    # Levenshtein ratio (normalized edit distance)
    features['name_levenshtein'] = (
        Levenshtein.ratio(name1, name2) if (name1 and name2) else 0.0)

    # Jaro-Winkler similarity
    features['name_jaro_winkler'] = (
        jellyfish.jaro_winkler_similarity(name1, name2) if (name1 and name2) else 0.0)

    # RapidFuzz token sort ratio (order-invariant)
    features['name_token_sort'] = fuzz.token_sort_ratio(name1, name2) / 100.0

    # RapidFuzz token set ratio (handles subset matches well)
    features['name_token_set'] = fuzz.token_set_ratio(name1, name2) / 100.0

    # RapidFuzz partial ratio (best substring match)
    features['name_partial'] = fuzz.partial_ratio(name1, name2) / 100.0

    # Jaccard similarity on name tokens
    features['name_jaccard'] = jaccard_similarity(name_tokens1, name_tokens2)

    # Overlap coefficient on name tokens
    features['name_overlap'] = overlap_coefficient(name_tokens1, name_tokens2)

    # Containment: how much of S1 name is in candidate
    features['name_containment'] = containment_score(name_tokens1, name_tokens2)

    # Name length ratio
    if name1 and name2:
        features['name_len_ratio'] = min(len(name1), len(name2)) / max(len(name1), len(name2))
    else:
        features['name_len_ratio'] = 0.0

    # Token count ratio
    n1 = len(name_tokens1) if name_tokens1 else 0
    n2 = len(name_tokens2) if name_tokens2 else 0
    if n1 > 0 and n2 > 0:
        features['name_token_count_ratio'] = min(n1, n2) / max(n1, n2)
    else:
        features['name_token_count_ratio'] = 0.0

    # Metaphone match
    mph1 = s1_row.get('name_metaphone', '')
    mph2 = cand_row.get('name_metaphone', '')
    features['name_metaphone_match'] = (
        1.0 if (mph1 and mph2 and mph1 == mph2) else 0.0)

    # ==========================================
    # ADDRESS FEATURES
    # ==========================================

    # Levenshtein ratio on address
    features['addr_levenshtein'] = (
        Levenshtein.ratio(addr1, addr2) if (addr1 and addr2) else 0.0)

    # Jaro-Winkler on address
    features['addr_jaro_winkler'] = (
        jellyfish.jaro_winkler_similarity(addr1, addr2) if (addr1 and addr2) else 0.0)

    # Token sort ratio on address
    features['addr_token_sort'] = fuzz.token_sort_ratio(addr1, addr2) / 100.0

    # Token set ratio on address
    features['addr_token_set'] = fuzz.token_set_ratio(addr1, addr2) / 100.0

    # Jaccard on address tokens
    features['addr_jaccard'] = jaccard_similarity(addr_tokens1, addr_tokens2)

    # Overlap on address tokens
    features['addr_overlap'] = overlap_coefficient(addr_tokens1, addr_tokens2)

    # Numeric overlap in addresses (street numbers, PIN codes, etc.)
    features['addr_num_jaccard'] = jaccard_similarity(addr_nums1, addr_nums2)
    features['addr_num_overlap'] = overlap_coefficient(addr_nums1, addr_nums2)

    # Address non-empty flags
    features['addr1_nonempty'] = 1.0 if addr1 else 0.0
    features['addr2_nonempty'] = 1.0 if addr2 else 0.0
    features['both_addr_nonempty'] = 1.0 if (addr1 and addr2) else 0.0

    # ==========================================
    # CROSS FEATURES
    # ==========================================

    # Country match
    features['country_match'] = (
        1.0 if s1_row['country_lower'] == cand_row['country_lower'] else 0.0)

    # Combined score (name + address)
    features['combined_levenshtein'] = (
        features['name_levenshtein'] * 0.6 + features['addr_levenshtein'] * 0.4)

    # Max of name similarities (catches cases where one metric works better)
    features['name_max_sim'] = max(
        features['name_levenshtein'],
        features['name_jaro_winkler'],
        features['name_token_sort'],
        features['name_token_set'],
    )

    # Max of address similarities
    features['addr_max_sim'] = max(
        features['addr_levenshtein'],
        features['addr_jaro_winkler'],
        features['addr_token_sort'],
        features['addr_token_set'],
    )

    # Source indicator (S2 vs S3)
    cand_id = cand_row['entity_id']
    features['is_s2'] = 1.0 if cand_id.startswith('S2-') else 0.0
    features['is_s3'] = 1.0 if cand_id.startswith('S3-') else 0.0

    return features


# Feature names (order matters for XGBoost)
FEATURE_NAMES = list(compute_pair_features(
    train_s1.iloc[0], train_s2.iloc[0]
).keys())
print(f'Number of features: {len(FEATURE_NAMES)}')
print(f'Features: {FEATURE_NAMES}')

## 7. Build Training Dataset (Positive + Negative Pairs)

In [ ]:
def build_training_pairs(s1_df, candidates_df, candidate_map, gt_map,
                         neg_ratio=3):
    """Build feature matrix and labels from candidate pairs.

    For each S1 entity:
      - All true matches found in candidates -> positive pairs
      - Random non-matching candidates -> negative pairs (up to neg_ratio x positives)
    """

    # Build lookup dicts for fast access
    s1_dict = {row['entity_id']: row for _, row in s1_df.iterrows()}
    cand_dict = {row['entity_id']: row for _, row in candidates_df.iterrows()}

    all_features = []
    all_labels = []

    for s1_id in tqdm(candidate_map, desc='Building pairs'):
        if s1_id not in s1_dict:
            continue

        s1_row = s1_dict[s1_id]
        true_matches = gt_map.get(s1_id, set())
        cands = candidate_map[s1_id]

        # Positive pairs (true matches that are in candidates)
        positives = cands & true_matches
        for cand_id in positives:
            if cand_id in cand_dict:
                feats = compute_pair_features(s1_row, cand_dict[cand_id])
                all_features.append([feats[f] for f in FEATURE_NAMES])
                all_labels.append(1)

        # Also add true matches not in candidates as positives (hard examples)
        missed_positives = true_matches - cands
        for cand_id in list(missed_positives)[:2]:  # limit to avoid imbalance
            if cand_id in cand_dict:
                feats = compute_pair_features(s1_row, cand_dict[cand_id])
                all_features.append([feats[f] for f in FEATURE_NAMES])
                all_labels.append(1)

        # Negative pairs (non-matching candidates)
        negatives = cands - true_matches
        n_neg = min(len(negatives),
                    max(neg_ratio * max(len(positives), 1), 3))
        neg_sample = (
            np.random.choice(list(negatives),
                             size=min(n_neg, len(negatives)),
                             replace=False)
            if negatives else [])

        for cand_id in neg_sample:
            if cand_id in cand_dict:
                feats = compute_pair_features(s1_row, cand_dict[cand_id])
                all_features.append([feats[f] for f in FEATURE_NAMES])
                all_labels.append(0)

    X = np.array(all_features, dtype=np.float32)
    y = np.array(all_labels, dtype=np.int32)

    print(f'\nTraining pairs: {len(y):,}')
    print(f'  Positive: {(y == 1).sum():,} ({(y == 1).mean():.1%})')
    print(f'  Negative: {(y == 0).sum():,} ({(y == 0).mean():.1%})')

    return X, y

In [ ]:
%%time
# Combine S2 and S3 for lookup
train_all_cands = pd.concat([train_s2, train_s3], ignore_index=True)

print('Building training pairs...')
X_train_full, y_train_full = build_training_pairs(
    train_s1_sample, train_all_cands, train_candidates, gt_map, neg_ratio=3
)
print(f'\nFeature matrix shape: {X_train_full.shape}')

## 8. Train XGBoost Model

In [ ]:
def compute_f_beta(y_true, y_pred, beta=0.5):
    """Compute F_beta score."""
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    if precision + recall == 0:
        return 0.0

    f_beta = (1 + beta**2) * precision * recall / (beta**2 * precision + recall)
    return f_beta


def compute_macro_f05(predictions, gt):
    """Compute macro-averaged F_0.5 score across S1 entities."""
    scores = []
    for s1_id in gt:
        true_matches = gt[s1_id]
        pred_matches = predictions.get(s1_id, set())

        # Singleton handling
        if len(true_matches) == 0 and len(pred_matches) == 0:
            scores.append(1.0)  # Correctly predicted no match
            continue
        if len(true_matches) == 0 and len(pred_matches) > 0:
            scores.append(0.0)  # False matches on singleton
            continue
        if len(pred_matches) == 0 and len(true_matches) > 0:
            scores.append(0.0)  # Missed all matches
            continue

        tp = len(true_matches & pred_matches)
        fp = len(pred_matches - true_matches)
        fn = len(true_matches - pred_matches)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0

        if precision + recall == 0:
            scores.append(0.0)
        else:
            f05 = 1.25 * precision * recall / (0.25 * precision + recall)
            scores.append(f05)

    return np.mean(scores)

print('Scoring functions defined.')

In [ ]:
# Split into train/val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42,
    stratify=y_train_full
)

print(f'Train: {X_train.shape[0]:,} pairs')
print(f'Val:   {X_val.shape[0]:,} pairs')
print(f'Train positive rate: {y_train.mean():.3f}')
print(f'Val positive rate:   {y_val.mean():.3f}')

In [ ]:
%%time
# Calculate scale_pos_weight for class imbalance
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

print(f'scale_pos_weight: {scale_pos_weight:.2f}')

# XGBoost model - tuned for precision (F_0.5 is precision-heavy)
model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    use_label_encoder=False,
    tree_method='hist',  # Fast for large data; use 'gpu_hist' if GPU available
    random_state=42,
    n_jobs=-1,
)

# Check if GPU is available
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if result.returncode == 0:
        model.set_params(tree_method='gpu_hist', device='cuda')
        print('Using GPU acceleration!')
except FileNotFoundError:
    print('No GPU detected, using CPU.')

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

print('\nModel training complete!')

In [ ]:
# Find optimal threshold for F_0.5
val_probs = model.predict_proba(X_val)[:, 1]

best_f05 = 0
best_threshold = 0.5

for threshold in np.arange(0.3, 0.9, 0.01):
    val_preds = (val_probs >= threshold).astype(int)
    f05 = compute_f_beta(y_val, val_preds, beta=0.5)
    if f05 > best_f05:
        best_f05 = f05
        best_threshold = threshold

print(f'Best threshold: {best_threshold:.2f}')
print(f'Best pair-level F_0.5: {best_f05:.4f}')

# Report precision/recall at best threshold
val_preds = (val_probs >= best_threshold).astype(int)
tp = np.sum((val_preds == 1) & (y_val == 1))
fp = np.sum((val_preds == 1) & (y_val == 0))
fn = np.sum((val_preds == 0) & (y_val == 1))
print(f'Precision: {tp/(tp+fp):.4f}')
print(f'Recall:    {tp/(tp+fn):.4f}')

In [ ]:
# Feature importance
import matplotlib.pyplot as plt

importance = model.feature_importances_
sorted_idx = np.argsort(importance)[::-1]

plt.figure(figsize=(12, 8))
plt.barh(range(len(FEATURE_NAMES)),
         importance[sorted_idx[::-1]],
         align='center')
plt.yticks(range(len(FEATURE_NAMES)),
           [FEATURE_NAMES[i] for i in sorted_idx[::-1]])
plt.xlabel('Feature Importance (Gain)')
plt.title('XGBoost Feature Importances')
plt.tight_layout()
plt.show()

print('\nTop 10 features:')
for i in sorted_idx[:10]:
    print(f'  {FEATURE_NAMES[i]:30s} {importance[i]:.4f}')

## 9. Validate on Training Holdout (Entity-Level F0.5)

In [ ]:
def predict_matches(s1_df, candidates_df, candidate_map, model, threshold):
    """Run the model on all candidate pairs and return predicted matches."""

    s1_dict = {row['entity_id']: row for _, row in s1_df.iterrows()}
    cand_dict = {row['entity_id']: row for _, row in candidates_df.iterrows()}

    predictions = {}

    for s1_id in tqdm(s1_df['entity_id'], desc='Predicting'):
        predictions[s1_id] = set()  # Default: no matches (singleton)

        if s1_id not in candidate_map or not candidate_map[s1_id]:
            continue

        s1_row = s1_dict[s1_id]
        cands = candidate_map[s1_id]

        # Batch compute features
        batch_features = []
        batch_ids = []
        for cand_id in cands:
            if cand_id in cand_dict:
                feats = compute_pair_features(s1_row, cand_dict[cand_id])
                batch_features.append([feats[f] for f in FEATURE_NAMES])
                batch_ids.append(cand_id)

        if not batch_features:
            continue

        X = np.array(batch_features, dtype=np.float32)
        probs = model.predict_proba(X)[:, 1]

        for i, cand_id in enumerate(batch_ids):
            if probs[i] >= threshold:
                predictions[s1_id].add(cand_id)

    return predictions

print('Prediction function defined.')

In [ ]:
%%time
# Validation: split sampled S1 entities into train/val for entity-level evaluation
sampled_s1_ids = list(train_s1_sample['entity_id'].values)
np.random.seed(42)
np.random.shuffle(sampled_s1_ids)
val_split = int(len(sampled_s1_ids) * 0.2)
val_s1_ids = set(sampled_s1_ids[:val_split])

val_s1_df = train_s1_sample[train_s1_sample['entity_id'].isin(val_s1_ids)]
val_candidates = {k: v for k, v in train_candidates.items() if k in val_s1_ids}
val_gt = {k: v for k, v in gt_map.items() if k in val_s1_ids}

print(f'Validation S1 entities: {len(val_s1_df):,}')
print(f'Validation entities with candidates: {len(val_candidates):,}')

# Run predictions
val_predictions = predict_matches(
    val_s1_df, train_all_cands, val_candidates, model, best_threshold
)

# Compute entity-level macro F_0.5
macro_f05 = compute_macro_f05(val_predictions, val_gt)
print(f'\n{"="*50}')
print(f'Entity-Level Macro F_0.5: {macro_f05:.4f}')
print(f'{"="*50}')

# Breakdown by country
for country in val_s1_df['country_lower'].unique():
    country_ids = set(
        val_s1_df[val_s1_df['country_lower'] == country]['entity_id'])
    country_gt = {k: v for k, v in val_gt.items() if k in country_ids}
    country_preds = {
        k: v for k, v in val_predictions.items() if k in country_ids}
    country_f05 = compute_macro_f05(country_preds, country_gt)
    print(f'  {country}: F_0.5 = {country_f05:.4f} ({len(country_ids):,} entities)')

## 10. Retrain on Full Training Sample

In [ ]:
%%time
# Retrain on full training data (no val split)
print('Retraining on full training sample...')

model_final = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    use_label_encoder=False,
    tree_method='hist',
    random_state=42,
    n_jobs=-1,
)

# Use GPU if available
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if result.returncode == 0:
        model_final.set_params(tree_method='gpu_hist', device='cuda')
except (FileNotFoundError, NameError):
    pass

model_final.fit(X_train_full, y_train_full, verbose=50)
print('Final model trained!')

# Save model
model_final.save_model('/content/xgb_entity_resolution.json')
print('Model saved to /content/xgb_entity_resolution.json')

## 11. Generate Test Predictions

Run the full pipeline on test data:
1. Build blocking index for test S2+S3
2. Generate candidate pairs
3. Predict matches using the trained model
4. Write output files

In [ ]:
%%time
# Build blocking index for test data
print('Building blocking index for test data...')
test_block_index = build_blocking_index(test_s2, test_s3)
print('Done!')

In [ ]:
%%time
# Run blocking on test data
# For the full test set, process in chunks to manage memory
CHUNK_SIZE = 50000  # Process this many S1 entities at a time

test_candidates = {}
s1_ids = test_s1['entity_id'].values
n_chunks = (len(s1_ids) + CHUNK_SIZE - 1) // CHUNK_SIZE

print(f'Processing {len(s1_ids):,} test S1 entities in {n_chunks} chunks...')

for chunk_idx in range(n_chunks):
    start = chunk_idx * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, len(s1_ids))
    chunk_ids = s1_ids[start:end]

    chunk_df = test_s1[test_s1['entity_id'].isin(chunk_ids)]

    print(f'\n--- Chunk {chunk_idx+1}/{n_chunks} ({len(chunk_df):,} entities) ---')
    chunk_candidates = multi_strategy_blocking(
        chunk_df, test_s2, test_s3, test_block_index
    )
    test_candidates.update(chunk_candidates)

    # Progress
    total_pairs = sum(len(v) for v in test_candidates.values())
    print(f'Running total: {total_pairs:,} candidate pairs')

    gc.collect()

print(f'\n=== Final: {sum(len(v) for v in test_candidates.values()):,} candidate pairs ===')

In [ ]:
%%time
# Predict matches on test data
test_all_cands = pd.concat([test_s2, test_s3], ignore_index=True)

print('Running predictions on test data...')
test_predictions = predict_matches(
    test_s1, test_all_cands, test_candidates, model_final, best_threshold
)

n_with_matches = sum(1 for v in test_predictions.values() if len(v) > 0)
n_singletons = sum(1 for v in test_predictions.values() if len(v) == 0)
print(f'\nEntities with matches: {n_with_matches:,}')
print(f'Singletons (no match): {n_singletons:,}')
print(f'Total predictions: {sum(len(v) for v in test_predictions.values()):,}')

## 12. Write Output Files

In [ ]:
def write_matching_results(predictions, all_s1_ids, output_path):
    """Write matching_results.tsv."""
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('source1_entity_id\tmatched_entity_ids\n')
        for s1_id in all_s1_ids:
            matches = predictions.get(s1_id, set())
            matched_str = ','.join(sorted(matches)) if matches else ''
            f.write(f'{s1_id}\t{matched_str}\n')
    print(f'Written {len(all_s1_ids):,} rows to {output_path}')


def write_candidate_pairs(candidates, all_s1_ids, output_path):
    """Write candidate_pairs.tsv."""
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('source1_entity_id\tcandidate_entity_ids\n')
        for s1_id in all_s1_ids:
            cands = candidates.get(s1_id, set())
            cand_str = ','.join(sorted(cands)) if cands else ''
            f.write(f'{s1_id}\t{cand_str}\n')
    print(f'Written {len(all_s1_ids):,} rows to {output_path}')


# Write outputs
all_test_s1_ids = list(test_s1['entity_id'].values)

matching_path = os.path.join(OUTPUT_DIR, 'matching_results.tsv')
candidate_path = os.path.join(OUTPUT_DIR, 'candidate_pairs.tsv')

write_matching_results(test_predictions, all_test_s1_ids, matching_path)
write_candidate_pairs(test_candidates, all_test_s1_ids, candidate_path)

print('\nOutput files written!')

In [ ]:
# Verify output format
print('=== matching_results.tsv (first 10 lines) ===')
!head -10 {matching_path}
print(f'\nTotal lines: {sum(1 for _ in open(matching_path)):,}')

print('\n=== candidate_pairs.tsv (first 10 lines) ===')
!head -10 {candidate_path}
print(f'\nTotal lines: {sum(1 for _ in open(candidate_path)):,}')

## 13. Validate Submission

In [ ]:
# Point this at validate_submission.py if you have it locally (e.g. inside
# student_resource/utils/). This cell is skipped safely if it isn't found.
VALIDATE_SCRIPT = 'student_resource/utils/validate_submission.py'

if os.path.exists(VALIDATE_SCRIPT):
    !python3 {VALIDATE_SCRIPT} \
        --matching {matching_path} \
        --candidate {candidate_path} \
        --test-dir {TEST_DIR}
else:
    print(f'Skipping validation: {VALIDATE_SCRIPT} not found.')


## 14. Download Outputs

In [ ]:
# Files are already saved locally under ./output - nothing to download.
print('Output files are in:', os.path.abspath(OUTPUT_DIR))
!ls -la {OUTPUT_DIR}


---

## Summary & Tips for Improvement

### Current Pipeline
| Stage | Method |
|-------|--------|
| Preprocessing | Transliteration (unidecode), legal suffix expansion, address normalization |
| Blocking | TF-IDF cosine (char 2-4 grams) + Metaphone + Name prefix, country-partitioned |
| Features | 30+ features: Levenshtein, Jaro-Winkler, token sort/set ratio, Jaccard, overlap, numeric overlap, metaphone match |
| Matching | XGBoost with F0.5-optimized threshold |

### Ways to Improve Score

1. **Increase `TRAIN_SAMPLE_SIZE`** - use more data for training (requires more RAM/time)
2. **Increase `TFIDF_TOP_K`** - more candidates = higher blocking recall, but slower
3. **Lower `TFIDF_MIN_SCORE`** - captures more candidates at cost of precision
4. **Add embedding features** - use a sentence transformer (<=8B params) for semantic similarity
5. **Hyperparameter tuning** - use Optuna to tune XGBoost params with F0.5 objective
6. **Better transliteration** - use `indic-transliteration` library for Hindi/Tamil/Kannada
7. **Address component parsing** - extract city/state/PIN separately for structured comparison
8. **Ensemble** - combine XGBoost with LightGBM or a neural model
9. **Transitive closure** - if A matches B and B matches C, consider A matches C
10. **Threshold per country** - different optimal thresholds for US, India, France